## Demo

Counterfactual explanations for event diagnosis in smart water systems.

This Python implementation is based on the [EPyT-Flow](https://github.com/WaterFutures/EPyT-Flow) package.

**DISCLAIMER:** This example/demo is not meant to be realistic! The aim is to demonstrate how to implement this specific concept in Python and how to use existing Python packages.

In [ ]:
%pip install epyt-flow
%pip install cvxpy

In [ ]:
!git clone https://github.com/andreArtelt/IAHR26ExplainableML4WDS.git

In [ ]:
import numpy as np
from epyt_flow.utils import plot_timeseries_data
import random
random.seed(4242)
np.random.seed(4242)

import sys
sys.path.insert(0,'/content/IAHR26ExplainableML4WDS/case-study-1')
from FaultDetection import EnsembleSystem, MyModel
from ensemble_consistent_counterfactuals import compute_ensemble_consistent_counterfactual
from cf_nun import NearestUnlikeNeighborCounterfactual
from generate_data import generate_data, pressure_sensor_locations
from utils import evaluate_fault_detection, plot_counterfactual

## Data generation

Create scenarios (21-days long) based on [Hanoi](https://waterfutures.github.io/WaterBenchmarkHub/benchmarks/network-Hanoi.html) from [LeakDB](https://waterfutures.github.io/WaterBenchmarkHub/benchmarks/KIOS-LeakDB.html) -- include random sensor faults (i.e., adding Gaussian noise).

<img src="Hanoi-topo.png" style="width:600px;height:400px;">

In [ ]:
# Generate data
X_train, y_train = generate_data(scenarios_id=["1", "2", "3"], apply_sensor_faults=False, apply_leakages=False)
X_test, y_test = generate_data(scenarios_id=["10"], apply_sensor_faults=True, apply_leakages=False)
print(X_train.shape)
print(X_test.shape)

In [ ]:
plot_timeseries_data(X_test.T, x_axis_label="Time steps (30min)", y_axis_label="Pressure (m)",
                     labels=[f"Node {s}" for s in pressure_sensor_locations])

## Process data -- apply time window

In [ ]:
def preprocess_data(X_data, y_labels, time_win_len=3, time_start=100):
    X_final = []
    Y_final = []
    y_faulty = []

    sensors_idx = list(range(X_data.shape[1]))

    # Use a sliding time window to construct a labeled data set
    t_index = time_start
    time_points = range(len(y_labels))
    i = 0
    while t_index < len(time_points) - time_win_len:
        # Grab time window from data stream
        x = X_data[t_index:t_index+time_win_len-1, :]

        #######################
        # Feature engineering #
        #######################
        x = np.mean(x, axis=0)  # "Stupid" feature
        X_final.append(x)

        Y_final.append([X_data[t_index + time_win_len-1, n] for n in sensors_idx])

        y_faulty.append(y_labels[t_index + time_win_len-1])

        t_index += 1  # Note: Overlapping time windows
        i += 1

    X_final = np.array(X_final)
    Y_final = np.array(Y_final)
    y_faulty = np.array(y_faulty)

    return X_final, Y_final, y_faulty

In [ ]:
# Pre-process data (i.e. applying a time-window, etc.)
time_window_lenth = 2
X_train_final, Y_train_final, _ = preprocess_data(X_train, y_train, time_win_len=time_window_lenth)
X_test_final, Y_test_final, y_test_faulty = preprocess_data(X_test, y_test, time_win_len=time_window_lenth)

## Fit and apply a simple event detector

Create a virtual sensors (linear regression) for each existing sensor. Raise an alarm whenever the prediction deviates too much from the observation:
$$||\vec{f}(\vec{x}) - \vec{y}(k)||_{\infty} > \theta$$
where $\vec{x}$ refers to the input of the virtual sensors -- i.e., previous sensor readings of all other sensors.

Note that this approach is a bit simplistic and can/should be enhanced for more complex sceanrios. However, it is performing sufficiently well for the purpose of demonstrating the potential of XAI, in particular counterfactual explanations.

In [ ]:
# Fit and evaluate event detector
detector = EnsembleSystem(MyModel, flow_nodes=[], pressure_nodes=list(range(X_train.shape[1])))
detector.fit(X_train_final, Y_train_final)

suspicious_time_points, _ = detector.apply_detector(X_test_final, Y_test_final)
faults_time = np.where(y_test_faulty == 1)[0]
print(evaluate_fault_detection(suspicious_time_points, faults_time, list(range(len(y_test_faulty)))))
if len(suspicious_time_points) == 0:
    print("No events detected!")

## Compute counterfactual explanations of detected events

We provide two implementations of methods for computing counterfactual explanation.

Computing counterfactuals by solving a convex qudratic optimization problem:

$$\underset{\vec{x}_{\text{cf}} \,\in\, \mathbb{R}^d}{\arg\min}\, |\vec{x}_{\text{cf}} - \vec{x}||_1 \\
\text{s.t. }  (\vec{w}_i^\top \vec{x}_{\text{cf}} + b_i - y_i)^2 \leq \delta_i + \xi_i \quad \forall \, i \\
\xi_i \geq 0  \quad \forall \, i$$

Nearest Unlike Neighbor (NUN) method, where the set of possible solutions (i.e., counterfactuals) is limited to a given set:

$$\underset{\vec{x}_{\text{cf}} \,\in\, \mathcal{D}}{\arg\min}\, ||\vec{x}_{\text{cf}} - \vec{x}||_1 \\
\text{s.t. } (\vec{w}_i^\top \vec{x}_{\text{cf}} + b_i - y_i)^2 \leq \delta_i + \xi_i \quad \forall \, i \\
\xi_i \geq 0  \quad \forall \, i$$

In [ ]:
# Use counterfactuals to explain event detections -- evaluate their cost!
models_wrapper = []
for m in detector.models:
    threshold = m["fault_detector"].threshold
    models_wrapper.append({"model": m["model"].wrap_model(), "feature_id_dropped": m["target_idx"], "threshold": threshold})

nun = NearestUnlikeNeighborCounterfactual(X_train)
event_counterfactuals_ = []
for t in suspicious_time_points:
    try:
        x_orig = X_test_final[t, :]
        y_orig = Y_test_final[t, :]

        #xcf = nun.compute_counterfactual(x_orig)   # NUN
        #delta_cf = xcf - x_orig

        xcf, delta_cf = compute_ensemble_consistent_counterfactual(models_wrapper, x_orig, y_orig)
        if xcf is None or delta_cf is None:
            continue

        # Store normalized changes
        delta_cf = np.abs(delta_cf) / np.sum(np.abs(delta_cf))
        event_counterfactuals_.append(delta_cf)
    except Exception as ex:
        print(ex)


In [ ]:
# Show counterfactual explanations
plot_counterfactual(event_counterfactuals_[0])